<a href="https://colab.research.google.com/github/RDGopal/IB9AU-2026/blob/main/Agents4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Context Protocol (MCP) with Agents

**The Future of Agentic AI Integration**

In this notebook, we explore **Model Context Protocol (MCP)**, a standardized way for AI agents to connect to external services.

## What We'll Build
1. Understand what MCP is and why it matters
2. Create MCP-compatible tools for financial analysis  
3. Build an agent that uses these tools
4. Compare MCP vs traditional approaches

## Why MCP?
- **Standardization**: One protocol for all integrations
- **Reusability**: Write tools once, use everywhere
- **Discoverability**: Agents can query available tools
- **Security**: Centralized access control

## Part 1: Setup

In [ ]:
# ==========================================
# Install Dependencies
# ==========================================
# We need 'accelerate' and 'bitsandbytes' to load the model efficiently on the GPU.
# Added 'duckduckgo_search' to enable web search capabilities for the agent.
!pip install -q smolagents transformers accelerate bitsandbytes yfinance seaborn matplotlib duckduckgo_search litellm

import os
from google.colab import userdata

# Set Hugging Face API key for LiteLLM, if available
# You'll need to add your Hugging Face API key to Colab Secrets as 'HF_TOKEN'
try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ["HUGGINGFACE_API_KEY"] = hf_token
        print("✅ Hugging Face API Key (HF_TOKEN) loaded from Colab Secrets.")
    else:
        print("⚠️ Hugging Face API Key (HF_TOKEN) not found in Colab Secrets. Please add it as 'HF_TOKEN' for Qwen models if needed.")
except Exception as e:
    print(f"⚠️ An error occurred while accessing Colab Secrets: {e}")
    print("⚠️ Hugging Face API Key (HF_TOKEN) not found in Colab Secrets. Please add it as 'HF_TOKEN' for Qwen models if needed.")

In [ ]:
# ==========================================
# Load the 3B Model (Lightweight & Fast)
# ==========================================
from smolagents import CodeAgent, TransformersModel
import torch

print("⬇️ Downloading Qwen 2.5 Coder 3B (approx 6GB)...")

# We use the 3B-Instruct model.
# device_map="auto" finds the Colab GPU.
# torch_dtype=torch.float16 cuts memory usage in half.
model = TransformersModel(
    model_id="Qwen/Qwen2.5-Coder-3B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16,
    max_new_tokens=2048  # Give it space to write longer scripts
)

print("✅ 3B Model loaded on GPU! Ready for coding.")

## Part 2: Understanding MCP

### Traditional Approach:
```
Agent → Function 1
     → Function 2  
     → Function 3
```
Every agent needs custom code.

### MCP Approach:
```
Agent ←→ MCP Client ←→ MCP Server ←→ Service
```
One standardized protocol, reusable everywhere.

We'll simulate MCP patterns using smolagents (true MCP requires separate server process, complex for Colab).

## Part 3: Building Financial MCP Server

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from typing import Dict, List

class FinancialMCPServer:
    """Simulates an MCP server for financial analysis."""

    def __init__(self):
        self.name = "financial-analysis-server"
        self.version = "1.0.0"

    def get_stock_data(self, ticker: str, period: str = "1y") -> Dict:
        """Get historical stock data and summary statistics."""
        try:
            stock = yf.Ticker(ticker)
            hist = stock.history(period=period)

            if hist.empty:
                return {"error": f"No data for {ticker}"}

            return {
                "ticker": ticker,
                "current_price": round(hist['Close'].iloc[-1], 2),
                "period_return_pct": round(((hist['Close'].iloc[-1] / hist['Close'].iloc[0]) - 1) * 100, 2),
                "volatility_pct": round(hist['Close'].pct_change().std() * np.sqrt(252) * 100, 2),
                "high_52w": round(hist['High'].max(), 2),
                "low_52w": round(hist['Low'].min(), 2)
            }
        except Exception as e:
            return {"error": str(e)}

    def calculate_sharpe_ratio(self, ticker: str, risk_free_rate: float = 0.04) -> Dict:
        """Calculate Sharpe Ratio for risk-adjusted returns."""
        try:
            stock = yf.Ticker(ticker)
            hist = stock.history(period="1y")
            returns = hist['Close'].pct_change().dropna()

            annual_return = returns.mean() * 252
            annual_vol = returns.std() * np.sqrt(252)
            sharpe = (annual_return - risk_free_rate) / annual_vol

            interpretation = (
                "Excellent" if sharpe > 2 else
                "Good" if sharpe > 1 else
                "Acceptable" if sharpe > 0 else "Poor"
            )

            return {
                "ticker": ticker,
                "sharpe_ratio": round(sharpe, 3),
                "annual_return_pct": round(annual_return * 100, 2),
                "annual_volatility_pct": round(annual_vol * 100, 2),
                "interpretation": interpretation
            }
        except Exception as e:
            return {"error": str(e)}

    def get_company_fundamentals(self, ticker: str) -> Dict:
        """Get key financial metrics."""
        try:
            stock = yf.Ticker(ticker)
            info = stock.info

            return {
                "ticker": ticker,
                "company": info.get('longName', 'N/A'),
                "sector": info.get('sector', 'N/A'),
                "market_cap": info.get('marketCap', 'N/A'),
                "pe_ratio": info.get('trailingPE', 'N/A'),
                "beta": info.get('beta', 'N/A'),
                "dividend_yield_pct": round(info.get('dividendYield', 0) * 100, 2) if info.get('dividendYield') else 0
            }
        except Exception as e:
            return {"error": str(e)}

    def check_diversification(self, tickers: List[str], period: str = "1y") -> Dict:
        """Analyze portfolio correlation."""
        try:
            data = yf.download(tickers, period=period, progress=False)['Close']
            returns = data.pct_change().dropna()
            corr_matrix = returns.corr()

            # Average correlation excluding diagonal
            mask = np.ones_like(corr_matrix, dtype=bool)
            np.fill_diagonal(mask, False)
            avg_corr = corr_matrix.where(mask).mean().mean()

            assessment = (
                "Excellent diversification" if avg_corr < 0.3 else
                "Good diversification" if avg_corr < 0.6 else
                "Poor diversification (high correlation)"
            )

            return {
                "tickers": tickers,
                "avg_correlation": round(avg_corr, 3),
                "assessment": assessment
            }
        except Exception as e:
            return {"error": str(e)}

# Create server instance
mcp_server = FinancialMCPServer()
print("✅ Financial MCP Server created")

## Part 4: Test MCP Tools Directly

In [ ]:
# Test the tools
print("📊 Apple Stock Data:")
print(mcp_server.get_stock_data("AAPL"))

print("\n📈 Tesla Sharpe Ratio:")
print(mcp_server.calculate_sharpe_ratio("TSLA"))

print("\n🏢 Microsoft Fundamentals:")
print(mcp_server.get_company_fundamentals("MSFT"))

## Part 5: Wrap Tools for Agent Use

In [ ]:
from smolagents import tool

# Use @tool decorator to make functions agent-compatible

@tool
def get_stock_data(ticker: str, period: str = "1y") -> dict:
    """Get historical stock data with current price, returns, and volatility.

    Args:
        ticker: Stock symbol like 'AAPL' or 'TSLA'
        period: Time period: '1mo', '3mo', '6mo', '1y', '2y', '5y'
    """
    return mcp_server.get_stock_data(ticker, period)

@tool
def calculate_sharpe_ratio(ticker: str, risk_free_rate: float = 0.04) -> dict:
    """Calculate Sharpe Ratio for risk-adjusted returns (higher is better).

    Args:
        ticker: Stock symbol
        risk_free_rate: Annual risk-free rate (default 4%)
    """
    return mcp_server.calculate_sharpe_ratio(ticker, risk_free_rate)

@tool
def get_company_fundamentals(ticker: str) -> dict:
    """Get company metrics: sector, market cap, P/E ratio, beta, dividend.

    Args:
        ticker: Stock symbol
    """
    return mcp_server.get_company_fundamentals(ticker)

@tool
def check_diversification(tickers: list) -> dict:
    """Check if portfolio stocks are well diversified (low correlation is good).

    Args:
        tickers: List of stock symbols like ['AAPL', 'MSFT', 'GOOGL']
    """
    return mcp_server.check_diversification(tickers)

mcp_tools = [
    get_stock_data,
    calculate_sharpe_ratio,
    get_company_fundamentals,
    check_diversification
]

print("✅ Tools ready for agent")
print(f"🔧 {len(mcp_tools)} tools available")

## Part 6: Create MCP-Enabled Agent

In [ ]:
from smolagents import CodeAgent, LiteLLMModel
import os # Ensure os is imported for environment variables

model = LiteLLMModel(
    model_id="huggingface/Qwen/Qwen2.5-Coder-3B-Instruct",
    api_key=os.environ["HUGGINGFACE_API_KEY"]
)

agent = CodeAgent(
    tools=mcp_tools,
    model=model,
    max_steps=4
)

print("✅ Financial Analysis Agent ready")
print("🤖 Can analyze stocks, risk, fundamentals, and diversification")

## Part 7: Test the Agent

In [ ]:
# Test 1: Simple query
result = agent.run("What's Apple's current price and year-over-year return?",stream=False)


In [ ]:
print(result)

In [ ]:
# Test 2: Risk analysis
result = agent.run(
    "Analyze Tesla (TSLA): get the Sharpe Ratio and company fundamentals. "
    "Is it a good investment from a risk-adjusted perspective?",stream=False
)

In [ ]:
print(result)

In [ ]:
# Test 3: Portfolio analysis
result = agent.run(
    "I have AAPL, MSFT, GOOGL, and AMZN. Check if my portfolio is well diversified.",stream=False
)

In [ ]:
print(result)

In [ ]:
# Test 4: Comparative analysis
result = agent.run(
    "Compare NVIDIA (NVDA) vs Intel (INTC): "
    "1) Year performance, 2) Sharpe Ratio, 3) Key fundamentals. "
    "Which is better for a conservative investor?",stream=False
)

In [ ]:
print(result)

## Key Takeaways

### What We Learned:

1. **MCP Concept**: Standardized protocol for agent-service communication
2. **Benefits**: Reusability, discoverability, maintainability, security
3. **Pattern**: Server (tools) ↔ Client ↔ Agent

### MCP vs Traditional:

**Traditional**: Each agent has custom tool code (duplication)
**MCP**: Tools in centralized server, all agents connect (reuse)

### When to Use MCP:

✅ Multiple agents need same tools  
✅ Enterprise integrations  
✅ Centralized security/auditing  
✅ Long-term maintainability

❌ Single simple agent  
❌ Rapid prototyping  
❌ Agent-specific tools

### Next Steps:
- Explore: https://modelcontextprotocol.io
